# SpecWise — RAG Pipeline Notebook

**Scope of this notebook:**
- 2.1 Load & Inspect
- 2.2 Chunking Strategy
- 2.3 Embeddings & Vector Store (persisted to disk)


In [ ]:
import os
import re
import json
from pathlib import Path

import chromadb
from sentence_transformers import SentenceTransformer

try:
    from pypdf import PdfReader
    PYPDF_AVAILABLE = True
except ImportError:
    PYPDF_AVAILABLE = False

DOCS_DIR = Path("../data/raw_docs/library-management-system")
VECTOR_STORE_DIR = Path("../backend/data/vector_store")
PROJECT_ID = "library_management_demo"         
COLLECTION_NAME = f"project_{PROJECT_ID}"

CHUNK_SIZE_WORDS = 400  
CHUNK_OVERLAP_WORDS = 60
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"

print("Docs dir exists:", DOCS_DIR.exists())
print("Vector store dir:", VECTOR_STORE_DIR.resolve())


Docs dir exists: True
Vector store dir: D:\ITI AI level2\1. Final Project\SpecWise-RAG\backend\data\vector_store


## 2.1 Load & Inspect

In [2]:
def load_text_file(path: Path) -> str:
    return path.read_text(encoding="utf-8")


def load_pdf_file(path: Path) -> str:
    if not PYPDF_AVAILABLE:
        raise RuntimeError("pypdf not installed — run `uv add pypdf`")
    reader = PdfReader(str(path))
    text_parts = [page.extract_text() or "" for page in reader.pages]
    return "\n".join(text_parts)


def load_document(path: Path) -> str:
    suffix = path.suffix.lower()
    if suffix in (".md", ".txt"):
        return load_text_file(path)
    elif suffix == ".pdf":
        return load_pdf_file(path)
    else:
        raise ValueError(f"Unsupported file type: {suffix}")


raw_documents = {}   # doc_name -> raw text
load_report = []      # one row per file, for the markdown report below

for path in sorted(DOCS_DIR.glob("*")):
    if path.is_dir():
        continue
    row = {"file": path.name, "format": path.suffix.lower(), "status": "ok", "note": ""}
    try:
        text = load_document(path)
        if not text.strip():
            row["status"] = "empty"
            row["note"] = "No extractable text — likely a scanned/image-only file, needs OCR."
        else:
            raw_documents[path.name] = text
            row["note"] = f"{len(text.split())} words"
    except Exception as e:
        row["status"] = "failed"
        row["note"] = str(e)
    load_report.append(row)

for row in load_report:
    print(f"[{row['status'].upper():6}] {row['file']:20} {row['format']:6} {row['note']}")

print(f"\nLoaded {len(raw_documents)} of {len(load_report)} files successfully.")


[OK    ] API_Spec.md          .md    250 words
[OK    ] SRS.md               .md    360 words
[OK    ] UseCases.md          .md    369 words

Loaded 3 of 3 files successfully.


**Load & Inspect — report**

All 3 sample files (`SRS.md`, `UseCases.md`, `API_Spec.md`) parsed cleanly with no OCR or encoding issues — they're plain Markdown, so `pypdf` wasn't exercised on this run. The loader does support `.pdf` (via `pypdf`) for when real project documents include scanned or exported PDFs; any file that comes back with no extractable text is flagged as `empty` above rather than silently skipped, so a scanned SRS wouldn't disappear from the corpus without a visible warning.

Word counts per file were in the 250–450 range, small enough that (as seen in 2.2 below) most sections fit inside a single chunk without needing to split further.

## 2.2 Chunking Strategy

**Approach:** split each document on its Markdown headers (`#`, `##`, `###`) first, so a chunk boundary lines up with a section boundary rather than cutting a requirement in half. If a section is still longer than the target chunk size, it's further split with a sliding word-window and overlap.

**Chosen size: ~400 words per chunk, 60-word overlap.**

**Justification:**
- 400 words sits in the middle of the assignment's recommended ~300–500 token range. Words and tokens aren't identical (roughly 1 word ≈ 1.3 tokens for English), so 400 words ≈ ~520 tokens — still comfortably inside a reasonable retrieval-chunk size without being so large that irrelevant sentences dilute the embedding.
- Requirements documents like an SRS are naturally organized into short, self-contained sections (a few sentences each, as seen in `SRS.md` §3.1–3.6). Splitting on headers first means most real sections become exactly one chunk — the retrieval unit matches the unit a human would cite ("per SRS §3.4"), which is exactly the citation granularity the assignment expects.
- 60-word overlap (~15% of chunk size) exists for the minority of sections that *do* exceed 400 words: it prevents a requirement's second half from losing the sentence that introduced it, at the cost of some duplicate content in the store. Because most sample-project sections are short, overlap barely gets exercised here — it will matter more on denser real project docs.

In [3]:
HEADER_PATTERN = re.compile(r"^(#{1,3})\s+(.*)$", re.MULTILINE)


def split_into_sections(text: str):
    """Split a markdown doc into (heading, section_text) pairs using # / ## / ### headers.
    Falls back to treating the whole doc as one section if no headers are found.
    """
    matches = list(HEADER_PATTERN.finditer(text))
    if not matches:
        return [(None, text.strip())]
    sections = []
    for i, m in enumerate(matches):
        heading = m.group(2).strip()
        start = m.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        body = text[start:end].strip()
        if body:
            sections.append((heading, body))
    return sections


def sliding_word_windows(words, size, overlap):
    """Slide a window of `size` words over `words`, stepping by (size - overlap)."""
    if len(words) <= size:
        return [words]
    windows = []
    step = size - overlap
    for start in range(0, len(words), step):
        window = words[start:start + size]
        if not window:
            break
        windows.append(window)
        if start + size >= len(words):
            break
    return windows


def chunk_document(doc_name: str, text: str):
    chunks = []
    for heading, body in split_into_sections(text):
        words = body.split()
        if len(words) <= CHUNK_SIZE_WORDS:
            chunks.append({"doc_name": doc_name, "section": heading, "text": body, "n_words": len(words)})
        else:
            for window in sliding_word_windows(words, CHUNK_SIZE_WORDS, CHUNK_OVERLAP_WORDS):
                chunks.append({"doc_name": doc_name, "section": heading, "text": " ".join(window), "n_words": len(window)})
    return chunks


all_chunks = []
for doc_name, text in raw_documents.items():
    doc_chunks = chunk_document(doc_name, text)
    all_chunks.extend(doc_chunks)
    print(f"{doc_name}: {len(doc_chunks)} chunks")

# stable chunk ids: <doc_name>::<index>
for i, c in enumerate(all_chunks):
    c["chunk_id"] = f"{c['doc_name']}::{i}"

print(f"\nTotal chunks: {len(all_chunks)}")
print(f"Avg words/chunk: {sum(c['n_words'] for c in all_chunks) / len(all_chunks):.1f}")
print(f"Max words/chunk: {max(c['n_words'] for c in all_chunks)}")
print(f"Min words/chunk: {min(c['n_words'] for c in all_chunks)}")


API_Spec.md: 6 chunks
SRS.md: 6 chunks
UseCases.md: 6 chunks

Total chunks: 18
Avg words/chunk: 49.0
Max words/chunk: 72
Min words/chunk: 29


In [4]:
# Inspect one chunk to sanity-check boundaries line up with a real requirement
sample = next(c for c in all_chunks if c["section"] and "Reservation" in c["section"])
print(f"doc = {sample['doc_name']}")
print(f"section = {sample['section']}")
print(f"chunk_id = {sample['chunk_id']}")
print("---")
print(sample["text"])


doc = SRS.md
section = 3.4 Reservation Rules
chunk_id = SRS.md::9
---
If a book is currently checked out, a member may place a reservation on it. Yes, a member may reserve a book that is already checked out; the system shall notify the reserving member by email within 24 hours of the book's return. Reservations expire automatically after 3 days if the reserved book is not picked up. A member may hold no more than 3 active reservations at once.


## 2.3 Embeddings & Vector Store

We embed every chunk with `sentence-transformers` (`all-MiniLM-L6-v2` — small, fast, good enough for this scale) and store the vectors in a **persisted** Chroma collection, one collection per `project_id` (`project_library_management_demo` here), matching the isolation model in Section 4 of the plan.

> **Note on running this cell:** it downloads the embedding model from Hugging Face Hub the first time. That requires normal internet access on your machine — run this notebook locally or in an environment with unrestricted network access, not in a network-locked sandbox.

In [5]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

texts = [c["text"] for c in all_chunks]
embeddings = embedding_model.encode(texts, show_progress_bar=True).tolist()

print(f"Embedded {len(embeddings)} chunks, dimension = {len(embeddings[0])}")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedded 18 chunks, dimension = 384


In [6]:
VECTOR_STORE_DIR.mkdir(parents=True, exist_ok=True)

chroma_client = chromadb.PersistentClient(path=str(VECTOR_STORE_DIR))
collection = chroma_client.get_or_create_collection(name=COLLECTION_NAME)

collection.add(
    ids=[c["chunk_id"] for c in all_chunks],
    embeddings=embeddings,
    documents=[c["text"] for c in all_chunks],
    metadatas=[{"doc_name": c["doc_name"], "section": c["section"] or ""} for c in all_chunks],
)

print(f"Collection '{COLLECTION_NAME}' now has {collection.count()} chunks persisted at {VECTOR_STORE_DIR.resolve()}")


Collection 'project_library_management_demo' now has 18 chunks persisted at D:\ITI AI level2\1. Final Project\SpecWise-RAG\backend\data\vector_store


**Persistence check.** The most common mistake at this stage (flagged explicitly in the plan) is not persisting to disk and rebuilding embeddings every run. To prove persistence actually works, we open a **brand-new** Chroma client pointed at the same directory — simulating what the FastAPI backend will do at startup in Phase 4 — and run a similarity search without re-embedding anything.

In [7]:
# Fresh client + fresh collection handle, no re-use of the objects created above
fresh_client = chromadb.PersistentClient(path=str(VECTOR_STORE_DIR))
fresh_collection = fresh_client.get_collection(name=COLLECTION_NAME)
print(f"Reloaded collection count: {fresh_collection.count()}")

test_question = "Can a member reserve a book that's already checked out?"
query_embedding = embedding_model.encode([test_question]).tolist()

results = fresh_collection.query(query_embeddings=query_embedding, n_results=3)

for rank, (doc, meta, dist) in enumerate(zip(results["documents"][0], results["metadatas"][0], results["distances"][0]), start=1):
    print(f"\n#{rank}  distance={dist:.4f}  doc={meta['doc_name']}  section={meta['section']}")
    print(doc[:220])


Reloaded collection count: 18

#1  distance=0.5087  doc=SRS.md  section=3.4 Reservation Rules
If a book is currently checked out, a member may place a reservation on it. Yes, a member may reserve a book that is already checked out; the system shall notify the reserving member by email within 24 hours of the book'

#2  distance=0.8123  doc=UseCases.md  section=UC-3: Borrow a Book
Actor: Member.
Preconditions: Member is in good standing and has fewer than 5 items currently checked out.
Main flow: Member presents the book and library card at checkout (or scans both at a self-checkout kiosk). The sy

#3  distance=0.8981  doc=API_Spec.md  section=POST /reservations
Places a reservation on a book that is currently checked out.
Request body: member_id (string), book_id (string).
Response: 201 Created with the reservation's queue position, or 400 Bad Request if the member already hold
